# 🏥 Multi-Input IoT-Based Telemedicine System
### Real-Time Remote Patient Monitoring Platform

---
**System Overview:**
This notebook implements a full telemedicine monitoring platform that:
- Monitors **50 patients** simultaneously across 5 disease categories
- Receives IoT-style sensor data continuously
- Stores all data persistently (CSV + SQLite)
- Processes data **in parallel** using threading
- Validates every reading and assigns risk levels
- Generates medical recommendations
- Displays a live dashboard

## Section 1: Imports & Dependencies

In [ ]:
# ── Standard Library ──────────────────────────────────────────────────────────
import threading
import queue
import time
import random
import json
import csv
import sqlite3
import os
import datetime
from collections import defaultdict
from dataclasses import dataclass, asdict, field
from typing import Dict, List, Optional, Any
from enum import Enum
import warnings
warnings.filterwarnings('ignore')

# ── Data & Visualization ──────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from IPython.display import display, HTML, clear_output

# ── Setup ─────────────────────────────────────────────────────────────────────
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')
random.seed(42)
np.random.seed(42)

print('✅ All libraries imported successfully!')
print(f'📅 System Start Time: {datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")}')

## Section 2: System Configuration & Enumerations

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# ENUMERATIONS: Disease cases and Risk levels
# ─────────────────────────────────────────────────────────────────────────────

class CaseType(Enum):
    CARDIOVASCULAR = "Cardiovascular"
    DIABETIC       = "Diabetic"
    RESPIRATORY    = "Respiratory"
    FEVER          = "Fever/Infection"
    GENERAL        = "General Monitoring"

class RiskLevel(Enum):
    NORMAL   = "Normal"
    WARNING  = "Warning"
    CRITICAL = "Critical"

# ─────────────────────────────────────────────────────────────────────────────
# PARAMETER DEFINITIONS WITH VALID RANGES
# Ranges are based on published clinical reference values.
# ─────────────────────────────────────────────────────────────────────────────

PARAMETER_RANGES = {
    # Sensor            : (min_possible, max_possible, normal_low, normal_high, warning_low, warning_high)
    "heart_rate"        : (20,  220,  60,  100,  50,  120),   # BPM
    "blood_pressure_sys": (50,  250,  90,  120,  80,  140),   # mmHg systolic
    "blood_pressure_dia": (30,  150,  60,   80,  50,   90),   # mmHg diastolic
    "spo2"              : (50,  100,  95,  100,  90,  100),   # %
    "glucose_level"     : (30,  600,  70,  140,  60,  200),   # mg/dL
    "insulin_level"     : (0,   300,   2,   25,   1,   50),   # µU/mL
    "respiratory_rate"  : (4,    60,  12,   20,  10,   25),   # breaths/min
    "body_temperature"  : (32,   43, 36.1, 37.2, 35,  38),   # °C
}

# Parameters required per case type
CASE_PARAMETERS = {
    CaseType.CARDIOVASCULAR : ["heart_rate", "blood_pressure_sys", "blood_pressure_dia", "spo2"],
    CaseType.DIABETIC       : ["glucose_level", "heart_rate", "insulin_level", "body_temperature"],
    CaseType.RESPIRATORY    : ["spo2", "respiratory_rate", "body_temperature", "heart_rate"],
    CaseType.FEVER          : ["body_temperature", "heart_rate", "blood_pressure_sys", "respiratory_rate"],
    CaseType.GENERAL        : ["heart_rate", "body_temperature", "spo2", "blood_pressure_sys"],
}

# Distribution of 50 patients across disease groups
PATIENT_DISTRIBUTION = {
    CaseType.CARDIOVASCULAR : 12,
    CaseType.DIABETIC       : 10,
    CaseType.RESPIRATORY    : 10,
    CaseType.FEVER          : 10,
    CaseType.GENERAL        :  8,
}  # Total = 50

print('✅ System configuration loaded.')
print(f'   → Total patients : {sum(PATIENT_DISTRIBUTION.values())}')
print(f'   → Disease groups : {len(PATIENT_DISTRIBUTION)}')
print(f'   → Sensor types   : {len(PARAMETER_RANGES)}')

# Verify total = 50
assert sum(PATIENT_DISTRIBUTION.values()) == 50, "Patient count must be exactly 50!"
print('\n📋 Patient distribution:')
for case, count in PATIENT_DISTRIBUTION.items():
    print(f'   {case.value:<25} → {count} patients')

## Section 3: Data Models

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# DATA CLASSES: IoT Record & Patient State
# ─────────────────────────────────────────────────────────────────────────────

@dataclass
class IoTRecord:
    """Represents one incoming sensor reading — IoT-style format."""
    patient_id   : str
    case_type    : str
    sensor_type  : str
    timestamp    : str
    value        : float
    unit         : str
    # Validation results (filled during processing)
    is_valid     : bool  = True
    error_reason : str   = ""

@dataclass
class PatientState:
    """Tracks the live state and history for one patient."""
    patient_id   : str
    name         : str
    age          : int
    gender       : str
    case_type    : CaseType
    latest       : Dict[str, float] = field(default_factory=dict)
    abnormalities: List[str]        = field(default_factory=list)
    risk_level   : RiskLevel        = RiskLevel.NORMAL
    recommendation: str             = ""
    reading_count: int              = 0

# Units for each sensor type
SENSOR_UNITS = {
    "heart_rate"         : "BPM",
    "blood_pressure_sys" : "mmHg",
    "blood_pressure_dia" : "mmHg",
    "spo2"               : "%",
    "glucose_level"      : "mg/dL",
    "insulin_level"      : "µU/mL",
    "respiratory_rate"   : "br/min",
    "body_temperature"   : "°C",
}

print('✅ Data models defined.')

## Section 4: Patient Factory — Create 50 Patients

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# PATIENT FACTORY
# Generates realistic patient profiles distributed across disease groups.
# ─────────────────────────────────────────────────────────────────────────────

FIRST_NAMES = ["Ahmed","Mohamed","Omar","Ali","Hassan","Ibrahim","Youssef",
               "Sara","Nour","Layla","Fatima","Aya","Rania","Dina",
               "Khaled","Tarek","Mostafa","Amira","Hana","Nadia"]
LAST_NAMES  = ["Mahmoud","Hassan","Ibrahim","El-Sayed","Khalil","Mansour",
               "Farouk","Nasser","Rashid","Abdel-Rahman"]

def create_all_patients() -> Dict[str, PatientState]:
    """Create 50 patients distributed across all disease groups."""
    patients = {}
    pid = 1

    # Age ranges per disease (clinically informed)
    age_ranges = {
        CaseType.CARDIOVASCULAR : (45, 80),
        CaseType.DIABETIC       : (35, 75),
        CaseType.RESPIRATORY    : (20, 70),
        CaseType.FEVER          : (5,  65),
        CaseType.GENERAL        : (18, 90),
    }

    for case_type, count in PATIENT_DISTRIBUTION.items():
        for _ in range(count):
            patient_id = f"P{pid:03d}"
            age_lo, age_hi = age_ranges[case_type]
            gender = random.choice(["Male", "Female"])
            name = f"{random.choice(FIRST_NAMES)} {random.choice(LAST_NAMES)}"

            patients[patient_id] = PatientState(
                patient_id  = patient_id,
                name        = name,
                age         = random.randint(age_lo, age_hi),
                gender      = gender,
                case_type   = case_type,
            )
            pid += 1

    return patients

PATIENTS = create_all_patients()

print(f'✅ Created {len(PATIENTS)} patients.')
print('\n📋 Sample patient roster:')
df_patients = pd.DataFrame([
    {"ID": p.patient_id, "Name": p.name, "Age": p.age,
     "Gender": p.gender, "Case": p.case_type.value}
    for p in PATIENTS.values()
])
display(df_patients.head(15).style.set_table_styles([
    {'selector': 'th', 'props': [('background-color', '#2c3e50'), ('color', 'white')]}
]))
print(f'   ... and {len(PATIENTS)-15} more patients.')

## Section 5: IoT Data Generator (Sensor Simulator)

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# IOT SENSOR SIMULATOR
# Generates realistic sensor readings with controlled anomaly injection.
# Anomaly probability is configurable — simulates real-world sensor behavior.
# ─────────────────────────────────────────────────────────────────────────────

class SensorSimulator:
    """Simulates continuous IoT sensor readings for all patients."""

    def __init__(self, anomaly_rate: float = 0.15):
        """
        anomaly_rate: probability (0–1) that a reading is out of normal range.
        15% gives realistic clinical monitoring data.
        """
        self.anomaly_rate = anomaly_rate

    def generate_value(self, sensor: str, inject_anomaly: bool = False) -> float:
        """Generate a sensor value — normal or anomalous."""
        r = PARAMETER_RANGES[sensor]  # (min_possible, max_possible, normal_low, normal_high, ...)
        min_p, max_p, norm_lo, norm_hi = r[0], r[1], r[2], r[3]

        if inject_anomaly:
            # 50% chance: too high; 50%: too low
            if random.random() < 0.5:
                val = random.uniform(norm_hi * 1.05, max_p)  # Above normal
            else:
                val = random.uniform(min_p, norm_lo * 0.95)  # Below normal
        else:
            # Normal value with small Gaussian noise
            mean = (norm_lo + norm_hi) / 2
            std  = (norm_hi - norm_lo) / 6  # ~99.7% within range
            val  = np.random.normal(mean, std)
            val  = max(min_p, min(max_p, val))

        return round(val, 2)

    def generate_records_for_patient(self, patient: PatientState) -> List[IoTRecord]:
        """Generate one full set of readings for a patient (all their sensors)."""
        records = []
        sensors = CASE_PARAMETERS[patient.case_type]
        ts = datetime.datetime.now().isoformat()

        for sensor in sensors:
            inject = random.random() < self.anomaly_rate
            value  = self.generate_value(sensor, inject_anomaly=inject)

            # Occasionally inject missing value (None → -1 sentinel)
            if random.random() < 0.02:   # 2% missing rate
                value = None

            records.append(IoTRecord(
                patient_id  = patient.patient_id,
                case_type   = patient.case_type.value,
                sensor_type = sensor,
                timestamp   = ts,
                value       = value,
                unit        = SENSOR_UNITS.get(sensor, ""),
            ))
        return records

SIMULATOR = SensorSimulator(anomaly_rate=0.15)

# Quick demo
demo_patient = list(PATIENTS.values())[0]
demo_records = SIMULATOR.generate_records_for_patient(demo_patient)
print(f'✅ Sensor simulator ready.')
print(f'\n🔬 Demo readings for {demo_patient.name} ({demo_patient.case_type.value}):')
for r in demo_records:
    print(f'   [{r.timestamp[:19]}]  {r.sensor_type:<25} = {r.value} {r.unit}')

## Section 6: Data Validation Engine

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# VALIDATION ENGINE
# Checks every incoming record for:
#   1. Missing value
#   2. Impossible (physically impossible) value
#   3. Out-of-range value
#   4. Wrong parameter for the patient's assigned case
# ─────────────────────────────────────────────────────────────────────────────

class ValidationEngine:

    def validate(self, record: IoTRecord, patient: PatientState) -> IoTRecord:
        """Validate a record in-place; set is_valid and error_reason."""

        # Rule 1 — Missing value
        if record.value is None:
            record.is_valid     = False
            record.error_reason = "MISSING_VALUE"
            return record

        # Rule 2 — Wrong parameter for this case type
        expected_sensors = CASE_PARAMETERS[patient.case_type]
        if record.sensor_type not in expected_sensors:
            record.is_valid     = False
            record.error_reason = f"WRONG_PARAM_FOR_CASE ({record.sensor_type} not in {patient.case_type.value})"
            return record

        # Rule 3 — Physically impossible value
        if record.sensor_type in PARAMETER_RANGES:
            min_p, max_p = PARAMETER_RANGES[record.sensor_type][:2]
            if not (min_p <= record.value <= max_p):
                record.is_valid     = False
                record.error_reason = f"IMPOSSIBLE_VALUE ({record.value} outside [{min_p},{max_p}])"
                return record

        # Rule 4 — Out of clinical range (warning — still valid but flagged)
        if record.sensor_type in PARAMETER_RANGES:
            _, _, norm_lo, norm_hi, warn_lo, warn_hi = PARAMETER_RANGES[record.sensor_type]
            if not (norm_lo <= record.value <= norm_hi):
                record.error_reason = "OUT_OF_NORMAL_RANGE"
                # Record remains valid but flagged for risk assessment

        return record

VALIDATOR = ValidationEngine()

# Demo validation
print('✅ Validation engine ready.')
print('\n🔍 Validation demo:')

test_cases = [
    IoTRecord("P001","Cardiovascular","heart_rate","now", None,    "BPM"),
    IoTRecord("P001","Cardiovascular","heart_rate","now", 300,     "BPM"),   # impossible
    IoTRecord("P001","Cardiovascular","heart_rate","now", 145,     "BPM"),   # out of range
    IoTRecord("P001","Cardiovascular","glucose_level","now", 80,   "mg/dL"), # wrong param
    IoTRecord("P001","Cardiovascular","heart_rate","now", 75,      "BPM"),   # normal
]

patient_cv = next(p for p in PATIENTS.values() if p.case_type == CaseType.CARDIOVASCULAR)
for tc in test_cases:
    VALIDATOR.validate(tc, patient_cv)
    status = '✅ VALID' if tc.is_valid else '❌ INVALID'
    print(f'   {status:<12} | {tc.sensor_type:<22} = {tc.value}  →  {tc.error_reason or "OK"}')

## Section 7: Risk Assessment & Medical Recommendations

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# RISK ASSESSMENT ENGINE
# Evaluates latest readings and assigns: Normal / Warning / Critical
# Also generates a clinical recommendation message.
# ─────────────────────────────────────────────────────────────────────────────

class RiskAssessor:

    # How far outside normal range triggers each level
    WARNING_THRESHOLD  = 0.10   # 10% beyond normal
    CRITICAL_THRESHOLD = 0.25   # 25% beyond normal

    # Absolute critical overrides (any reading triggers Critical)
    ABSOLUTE_CRITICAL = {
        "spo2"              : ("<", 85),
        "heart_rate"        : (">", 180),
        "heart_rate_low"    : ("<", 35),
        "body_temperature"  : (">", 40.5),
        "glucose_level"     : ("<", 50),
        "blood_pressure_sys": (">", 180),
    }

    RECOMMENDATIONS = {
        (CaseType.CARDIOVASCULAR, RiskLevel.NORMAL)   : "Continue current medication. Follow-up in 1 month.",
        (CaseType.CARDIOVASCULAR, RiskLevel.WARNING)  : "Review ECG. Consider adjusting antihypertensive dosage.",
        (CaseType.CARDIOVASCULAR, RiskLevel.CRITICAL) : "🚨 URGENT: Dispatch emergency team. Possible cardiac event.",
        (CaseType.DIABETIC,       RiskLevel.NORMAL)   : "Maintain diet plan. Check glucose twice daily.",
        (CaseType.DIABETIC,       RiskLevel.WARNING)  : "Adjust insulin dose. Notify endocrinologist.",
        (CaseType.DIABETIC,       RiskLevel.CRITICAL) : "🚨 URGENT: Risk of diabetic coma/ketoacidosis. Immediate intervention.",
        (CaseType.RESPIRATORY,    RiskLevel.NORMAL)   : "Continue bronchodilator therapy. Weekly spirometry.",
        (CaseType.RESPIRATORY,    RiskLevel.WARNING)  : "Increase O2 supplementation. Chest X-ray recommended.",
        (CaseType.RESPIRATORY,    RiskLevel.CRITICAL) : "🚨 URGENT: Severe respiratory distress. Prepare for intubation.",
        (CaseType.FEVER,          RiskLevel.NORMAL)   : "Antipyretics as needed. Monitor temperature every 4h.",
        (CaseType.FEVER,          RiskLevel.WARNING)  : "Blood cultures ordered. Broad-spectrum antibiotics consideration.",
        (CaseType.FEVER,          RiskLevel.CRITICAL) : "🚨 URGENT: Sepsis protocol activated. ICU admission required.",
        (CaseType.GENERAL,        RiskLevel.NORMAL)   : "Patient stable. Routine monitoring.",
        (CaseType.GENERAL,        RiskLevel.WARNING)  : "Investigate abnormal readings. Specialist referral.",
        (CaseType.GENERAL,        RiskLevel.CRITICAL) : "🚨 URGENT: Multiple abnormal parameters. Immediate medical evaluation.",
    }

    def assess(self, patient: PatientState) -> PatientState:
        """Compute risk level and recommendation from latest readings."""
        if not patient.latest:
            return patient

        abnormalities = []
        max_risk      = RiskLevel.NORMAL

        for sensor, value in patient.latest.items():
            if sensor not in PARAMETER_RANGES:
                continue
            _, _, norm_lo, norm_hi, _, _ = PARAMETER_RANGES[sensor]

            # ── Determine deviation ──────────────────────────────────────────
            if value < norm_lo:
                deviation = (norm_lo - value) / norm_lo if norm_lo != 0 else 0
                direction = "LOW"
            elif value > norm_hi:
                deviation = (value - norm_hi) / norm_hi if norm_hi != 0 else 0
                direction = "HIGH"
            else:
                continue  # Normal

            # ── Assign risk level from deviation ─────────────────────────────
            if deviation >= self.CRITICAL_THRESHOLD:
                risk = RiskLevel.CRITICAL
            elif deviation >= self.WARNING_THRESHOLD:
                risk = RiskLevel.WARNING
            else:
                risk = RiskLevel.WARNING  # Any out-of-range is at least warning

            abnormalities.append(f"{sensor} {direction} ({value})")
            if risk.value > max_risk.value:
                max_risk = risk

        # ── Absolute critical overrides ──────────────────────────────────────
        hr  = patient.latest.get("heart_rate")
        spo = patient.latest.get("spo2")
        tmp = patient.latest.get("body_temperature")
        glc = patient.latest.get("glucose_level")
        sbp = patient.latest.get("blood_pressure_sys")

        if (hr  and hr  > 180) or (hr  and hr  < 35): max_risk = RiskLevel.CRITICAL
        if (spo and spo < 85):                         max_risk = RiskLevel.CRITICAL
        if (tmp and tmp > 40.5):                       max_risk = RiskLevel.CRITICAL
        if (glc and glc < 50):                         max_risk = RiskLevel.CRITICAL
        if (sbp and sbp > 180):                        max_risk = RiskLevel.CRITICAL

        patient.abnormalities  = abnormalities
        patient.risk_level     = max_risk
        patient.recommendation = self.RECOMMENDATIONS.get(
            (patient.case_type, max_risk),
            "Consult attending physician."
        )
        return patient

RISK_ASSESSOR = RiskAssessor()
print('✅ Risk assessment engine ready.')

## Section 8: Storage System (CSV + SQLite)

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# STORAGE SYSTEM
# Saves every validated record to:
#   1. CSV file  (telemedicine_data.csv)
#   2. SQLite DB (telemedicine.db)
#   3. JSON log  (telemedicine_log.json)
# Thread-safe using a dedicated write lock.
# ─────────────────────────────────────────────────────────────────────────────

class StorageSystem:

    CSV_PATH  = "telemedicine_data.csv"
    DB_PATH   = "telemedicine.db"
    JSON_PATH = "telemedicine_log.json"

    def __init__(self):
        self._lock = threading.Lock()
        self._init_csv()
        self._init_db()
        self._json_buffer = []

    # ── CSV ──────────────────────────────────────────────────────────────────
    def _init_csv(self):
        with open(self.CSV_PATH, "w", newline="") as f:
            writer = csv.writer(f)
            writer.writerow([
                "patient_id","case_type","sensor_type",
                "timestamp","value","unit","is_valid","error_reason"
            ])

    def _write_csv(self, record: IoTRecord):
        with open(self.CSV_PATH, "a", newline="") as f:
            writer = csv.writer(f)
            writer.writerow([
                record.patient_id, record.case_type, record.sensor_type,
                record.timestamp, record.value, record.unit,
                record.is_valid, record.error_reason
            ])

    # ── SQLite ────────────────────────────────────────────────────────────────
    def _init_db(self):
        conn = sqlite3.connect(self.DB_PATH)
        c    = conn.cursor()
        c.execute('''
            CREATE TABLE IF NOT EXISTS readings (
                id           INTEGER PRIMARY KEY AUTOINCREMENT,
                patient_id   TEXT    NOT NULL,
                case_type    TEXT    NOT NULL,
                sensor_type  TEXT    NOT NULL,
                timestamp    TEXT    NOT NULL,
                value        REAL,
                unit         TEXT,
                is_valid     INTEGER,
                error_reason TEXT
            )
        ''')
        c.execute('''
            CREATE TABLE IF NOT EXISTS patient_risk (
                patient_id    TEXT PRIMARY KEY,
                case_type     TEXT,
                risk_level    TEXT,
                abnormalities TEXT,
                recommendation TEXT,
                last_updated  TEXT
            )
        ''')
        conn.commit()
        conn.close()

    def _write_db(self, record: IoTRecord):
        conn = sqlite3.connect(self.DB_PATH)
        conn.execute(
            "INSERT INTO readings VALUES (NULL,?,?,?,?,?,?,?,?)",
            (record.patient_id, record.case_type, record.sensor_type,
             record.timestamp, record.value, record.unit,
             int(record.is_valid), record.error_reason)
        )
        conn.commit()
        conn.close()

    def update_patient_risk(self, patient: PatientState):
        """Upsert the patient's latest risk snapshot."""
        conn = sqlite3.connect(self.DB_PATH)
        conn.execute(
            """INSERT OR REPLACE INTO patient_risk
               VALUES (?,?,?,?,?,?)""",
            (patient.patient_id, patient.case_type.value,
             patient.risk_level.value,
             "; ".join(patient.abnormalities),
             patient.recommendation,
             datetime.datetime.now().isoformat())
        )
        conn.commit()
        conn.close()

    # ── JSON ──────────────────────────────────────────────────────────────────
    def _write_json(self, record: IoTRecord):
        self._json_buffer.append(asdict(record))
        if len(self._json_buffer) % 100 == 0:  # Flush every 100 records
            self._flush_json()

    def _flush_json(self):
        with open(self.JSON_PATH, "w") as f:
            json.dump(self._json_buffer, f, indent=2)

    # ── Public interface ──────────────────────────────────────────────────────
    def save(self, record: IoTRecord):
        """Thread-safe save of one IoT record to all storage backends."""
        with self._lock:
            self._write_csv(record)
            self._write_db(record)
            self._write_json(record)

    def flush_all(self):
        with self._lock:
            self._flush_json()

STORAGE = StorageSystem()
print(f'✅ Storage system initialized.')
print(f'   → CSV  : {StorageSystem.CSV_PATH}')
print(f'   → DB   : {StorageSystem.DB_PATH}')
print(f'   → JSON : {StorageSystem.JSON_PATH}')

## Section 9: Concurrent Processing Engine (Threading)

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CONCURRENT PROCESSING ENGINE
# Architecture:
#   - One PRODUCER thread per patient (50 threads) — generates sensor data
#   - A shared QUEUE holds incoming records
#   - A pool of CONSUMER threads validates, stores, and assesses risk
#   - A MONITOR thread collects stats
# This simulates real IoT data streaming.
# ─────────────────────────────────────────────────────────────────────────────

class TeleMedicineSystem:

    def __init__(self, patients, simulator, validator, risk_assessor, storage,
                 num_rounds: int = 5, interval: float = 0.3, num_consumers: int = 8):
        """
        num_rounds   : How many data cycles to run
        interval     : Seconds between each patient's readings
        num_consumers: Worker threads for processing
        """
        self.patients       = patients
        self.simulator      = simulator
        self.validator      = validator
        self.risk_assessor  = risk_assessor
        self.storage        = storage
        self.num_rounds     = num_rounds
        self.interval       = interval
        self.num_consumers  = num_consumers

        self.record_queue   = queue.Queue(maxsize=1000)
        self._stop_event    = threading.Event()

        # Thread-safe stats
        self._stats_lock    = threading.Lock()
        self.stats = {
            "total_received"  : 0,
            "valid"           : 0,
            "invalid"         : 0,
            "critical"        : 0,
            "warning"         : 0,
            "normal"          : 0,
        }

    # ── Producer: one per patient ─────────────────────────────────────────────
    def _patient_producer(self, patient: PatientState):
        for _ in range(self.num_rounds):
            if self._stop_event.is_set():
                break
            records = self.simulator.generate_records_for_patient(patient)
            for rec in records:
                self.record_queue.put(rec)
            time.sleep(self.interval + random.uniform(0, 0.1))  # Jitter

    # ── Consumer: pool of workers ─────────────────────────────────────────────
    def _consumer_worker(self):
        while not self._stop_event.is_set() or not self.record_queue.empty():
            try:
                record  = self.record_queue.get(timeout=0.5)
                patient = self.patients[record.patient_id]

                # 1. Validate
                record = self.validator.validate(record, patient)

                # 2. Store (regardless of validity)
                self.storage.save(record)

                # 3. Update patient state (only valid readings)
                if record.is_valid:
                    patient.latest[record.sensor_type] = record.value
                    patient.reading_count += 1

                    # 4. Risk assessment
                    self.risk_assessor.assess(patient)
                    self.storage.update_patient_risk(patient)

                # 5. Update stats
                with self._stats_lock:
                    self.stats["total_received"] += 1
                    if record.is_valid:
                        self.stats["valid"] += 1
                    else:
                        self.stats["invalid"] += 1

                self.record_queue.task_done()
            except queue.Empty:
                continue

    # ── Run the system ────────────────────────────────────────────────────────
    def run(self):
        print(f'🚀 Starting TeleMedicine System...')
        print(f'   Patients : {len(self.patients)}')
        print(f'   Rounds   : {self.num_rounds}')
        print(f'   Consumers: {self.num_consumers} threads')
        print()

        # Launch consumers
        consumers = []
        for i in range(self.num_consumers):
            t = threading.Thread(target=self._consumer_worker,
                                 name=f"Consumer-{i}", daemon=True)
            t.start()
            consumers.append(t)

        # Launch producers (one per patient)
        producers = []
        for patient in self.patients.values():
            t = threading.Thread(target=self._patient_producer,
                                 args=(patient,),
                                 name=f"Producer-{patient.patient_id}",
                                 daemon=True)
            t.start()
            producers.append(t)

        # Wait for all producers to finish
        for t in producers:
            t.join()

        # Signal consumers to stop once queue is drained
        self._stop_event.set()
        self.record_queue.join()

        for t in consumers:
            t.join(timeout=2)

        # Update risk stats
        for patient in self.patients.values():
            rl = patient.risk_level
            if rl == RiskLevel.CRITICAL:
                self.stats["critical"] += 1
            elif rl == RiskLevel.WARNING:
                self.stats["warning"] += 1
            else:
                self.stats["normal"] += 1

        self.storage.flush_all()
        print('✅ Data collection complete!')
        return self.stats

print('✅ TeleMedicine concurrent engine defined.')

## Section 10: Run the System ▶

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# RUN — This cell executes the full 50-patient monitoring simulation.
# num_rounds=5 means each patient sends 5 batches of readings.
# Total expected records ≈ 50 patients × 4 sensors × 5 rounds = 1000 records
# ─────────────────────────────────────────────────────────────────────────────

start_time = time.time()

SYSTEM = TeleMedicineSystem(
    patients      = PATIENTS,
    simulator     = SIMULATOR,
    validator     = VALIDATOR,
    risk_assessor = RISK_ASSESSOR,
    storage       = STORAGE,
    num_rounds    = 5,      # Each patient sends 5 rounds of data
    interval      = 0.2,    # 200ms between rounds per patient
    num_consumers = 10,     # 10 parallel consumer threads
)

final_stats = SYSTEM.run()
elapsed = time.time() - start_time

print()
print('=' * 50)
print('📊 SYSTEM STATISTICS')
print('=' * 50)
print(f'  Runtime          : {elapsed:.1f} seconds')
print(f'  Total records    : {final_stats["total_received"]}')
print(f'  Valid records    : {final_stats["valid"]}')
print(f'  Invalid records  : {final_stats["invalid"]}')
print(f'  Throughput       : {final_stats["total_received"]/elapsed:.0f} rec/sec')
print()
print(f'  🟢 Normal  patients : {final_stats["normal"]}')
print(f'  🟡 Warning patients : {final_stats["warning"]}')
print(f'  🔴 Critical patients: {final_stats["critical"]}')

## Section 11: Patient Report Output (Telemedicine Display)

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# TELEMEDICINE OUTPUT
# Formatted report for every patient including:
#   - Patient ID & demographics
#   - Medical case type
#   - Latest parameter readings
#   - Detected abnormalities
#   - Risk level
#   - Recommendation message
# ─────────────────────────────────────────────────────────────────────────────

RISK_ICONS = {
    RiskLevel.NORMAL   : "🟢",
    RiskLevel.WARNING  : "🟡",
    RiskLevel.CRITICAL : "🔴",
}

def print_patient_report(patient: PatientState):
    icon = RISK_ICONS[patient.risk_level]
    print(f"{'─'*60}")
    print(f"{icon} Patient: {patient.name} [{patient.patient_id}]  "
          f"Age: {patient.age}  Gender: {patient.gender}")
    print(f"   Case Type    : {patient.case_type.value}")
    print(f"   Risk Level   : {patient.risk_level.value}")
    print(f"   Total Readings: {patient.reading_count}")
    print(f"   Latest Values:")
    for sensor, value in patient.latest.items():
        unit = SENSOR_UNITS.get(sensor, "")
        r = PARAMETER_RANGES.get(sensor, ())
        flag = ""
        if r and (value < r[2] or value > r[3]):
            flag = " ⚠️"
        print(f"     • {sensor:<25} : {value:>8.2f} {unit}{flag}")
    if patient.abnormalities:
        print(f"   Abnormalities : {', '.join(patient.abnormalities)}")
    print(f"   Recommendation: {patient.recommendation}")

# Print report for all patients
print("╔══════════════════════════════════════════════════════════╗")
print("║       🏥  TELEMEDICINE PATIENT REPORTS                   ║")
print("╚══════════════════════════════════════════════════════════╝")

for patient in PATIENTS.values():
    print_patient_report(patient)

print(f"{'─'*60}")
print(f"✅ Report generated for {len(PATIENTS)} patients.")

## Section 12: Data Analysis & Visualization

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# LOAD DATA FROM STORAGE FOR ANALYSIS
# ─────────────────────────────────────────────────────────────────────────────

df = pd.read_csv(StorageSystem.CSV_PATH)
print(f'📂 Loaded {len(df):,} records from storage.')
print(f'   Columns: {list(df.columns)}')
print()
display(df.head(10))
print()
print('📊 Data Summary:')
display(df.describe())

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# VISUALIZATION 1: Risk Level Distribution
# ─────────────────────────────────────────────────────────────────────────────

fig, axes = plt.subplots(2, 3, figsize=(18, 11))
fig.suptitle('🏥 TeleMedicine System — Analytics Dashboard', fontsize=16, fontweight='bold')

# Build summary DataFrame from patient states
summary = pd.DataFrame([
    {
        "patient_id"  : p.patient_id,
        "case_type"   : p.case_type.value,
        "risk_level"  : p.risk_level.value,
        "age"         : p.age,
        "gender"      : p.gender,
        "reading_count": p.reading_count,
        "abnormality_count": len(p.abnormalities),
    }
    for p in PATIENTS.values()
])

# ── Plot 1: Risk Distribution Pie ────────────────────────────────────────────
ax = axes[0, 0]
risk_counts = summary['risk_level'].value_counts()
colors = {'Normal': '#2ecc71', 'Warning': '#f39c12', 'Critical': '#e74c3c'}
pie_colors = [colors.get(r, 'gray') for r in risk_counts.index]
wedges, texts, autotexts = ax.pie(
    risk_counts.values, labels=risk_counts.index,
    autopct='%1.1f%%', colors=pie_colors,
    startangle=90, explode=[0.05]*len(risk_counts)
)
for at in autotexts:
    at.set_fontsize(11)
ax.set_title('Risk Level Distribution', fontsize=13, fontweight='bold')

# ── Plot 2: Patients per Case Type (Bar) ──────────────────────────────────────
ax = axes[0, 1]
case_counts = summary['case_type'].value_counts()
bars = ax.bar(range(len(case_counts)), case_counts.values,
              color=sns.color_palette('husl', len(case_counts)),
              edgecolor='white', linewidth=1.5)
ax.set_xticks(range(len(case_counts)))
ax.set_xticklabels([c.replace('/', '/\n') for c in case_counts.index], fontsize=9)
ax.set_ylabel('Patient Count')
ax.set_title('Patients per Disease Group', fontsize=13, fontweight='bold')
for bar, val in zip(bars, case_counts.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
            str(val), ha='center', va='bottom', fontweight='bold')

# ── Plot 3: Risk by Case Type (Stacked Bar) ───────────────────────────────────
ax = axes[0, 2]
pivot = summary.groupby(['case_type', 'risk_level']).size().unstack(fill_value=0)
pivot_colors = [colors.get(c, 'gray') for c in pivot.columns]
pivot.plot(kind='bar', ax=ax, color=pivot_colors, edgecolor='white',
           linewidth=1.2, legend=True)
ax.set_xticklabels(pivot.index, rotation=30, ha='right', fontsize=8)
ax.set_ylabel('Patient Count')
ax.set_title('Risk Levels per Disease Group', fontsize=13, fontweight='bold')
ax.legend(title='Risk', fontsize=8)

# ── Plot 4: Age Distribution by Risk Level (Box) ──────────────────────────────
ax = axes[1, 0]
risk_order = ['Normal', 'Warning', 'Critical']
data_by_risk = [summary[summary['risk_level'] == r]['age'].values for r in risk_order]
bp = ax.boxplot(data_by_risk, labels=risk_order, patch_artist=True)
for patch, color in zip(bp['boxes'], ['#2ecc71', '#f39c12', '#e74c3c']):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)
ax.set_ylabel('Age (years)')
ax.set_title('Age Distribution by Risk Level', fontsize=13, fontweight='bold')

# ── Plot 5: Invalid Records by Reason ─────────────────────────────────────────
ax = axes[1, 1]
invalid = df[df['is_valid'] == False]
if len(invalid) > 0:
    err_counts = invalid['error_reason'].apply(
        lambda x: x.split('(')[0].strip() if x else 'UNKNOWN'
    ).value_counts()
    ax.barh(range(len(err_counts)), err_counts.values,
            color=sns.color_palette('Reds_r', len(err_counts)))
    ax.set_yticks(range(len(err_counts)))
    ax.set_yticklabels(err_counts.index, fontsize=9)
    ax.set_xlabel('Count')
else:
    ax.text(0.5, 0.5, 'No invalid records!', ha='center', va='center',
            transform=ax.transAxes, fontsize=14)
ax.set_title('Validation Failures by Type', fontsize=13, fontweight='bold')

# ── Plot 6: Readings per Sensor Type ──────────────────────────────────────────
ax = axes[1, 2]
sensor_counts = df['sensor_type'].value_counts()
ax.bar(range(len(sensor_counts)), sensor_counts.values,
       color=sns.color_palette('Blues_r', len(sensor_counts)))
ax.set_xticks(range(len(sensor_counts)))
ax.set_xticklabels(sensor_counts.index, rotation=40, ha='right', fontsize=8)
ax.set_ylabel('Total Readings')
ax.set_title('Readings per Sensor Type', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.savefig('telemedicine_dashboard.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Dashboard saved as telemedicine_dashboard.png')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# VISUALIZATION 2: Sensor Value Distributions per Case
# ─────────────────────────────────────────────────────────────────────────────

fig, axes = plt.subplots(2, 4, figsize=(20, 10))
fig.suptitle('Sensor Reading Distributions with Clinical Reference Ranges',
             fontsize=15, fontweight='bold')

sensors_to_plot = list(PARAMETER_RANGES.keys())
df_valid = df[df['is_valid'] == True].copy()

for idx, sensor in enumerate(sensors_to_plot):
    ax  = axes[idx // 4][idx % 4]
    sub = df_valid[df_valid['sensor_type'] == sensor]['value'].dropna()

    if len(sub) == 0:
        ax.set_visible(False)
        continue

    ax.hist(sub, bins=30, color='steelblue', alpha=0.75, edgecolor='white')

    r = PARAMETER_RANGES[sensor]
    ax.axvline(r[2], color='#2ecc71', linestyle='--', linewidth=2, label=f'Normal lo ({r[2]})')
    ax.axvline(r[3], color='#e74c3c', linestyle='--', linewidth=2, label=f'Normal hi ({r[3]})')

    ax.set_title(sensor.replace('_', ' ').title(), fontsize=11, fontweight='bold')
    ax.set_xlabel(SENSOR_UNITS.get(sensor, ""), fontsize=9)
    ax.set_ylabel('Frequency')
    ax.legend(fontsize=7)

plt.tight_layout()
plt.savefig('sensor_distributions.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Sensor distributions saved.')

## Section 13: Database Query Examples

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# DATABASE QUERIES — Demonstrating SQL storage capability
# ─────────────────────────────────────────────────────────────────────────────

conn = sqlite3.connect(StorageSystem.DB_PATH)

print('━'*60)
print('🗄️  DATABASE QUERIES')
print('━'*60)

# Q1: Total records per case type
print('\n📌 Q1: Total records received per disease group:')
q1 = pd.read_sql_query(
    "SELECT case_type, COUNT(*) as total, SUM(is_valid) as valid "
    "FROM readings GROUP BY case_type ORDER BY total DESC",
    conn
)
display(q1)

# Q2: Critical patients
print('\n📌 Q2: All CRITICAL patients:')
q2 = pd.read_sql_query(
    "SELECT patient_id, case_type, risk_level, abnormalities, recommendation "
    "FROM patient_risk WHERE risk_level = 'Critical'",
    conn
)
display(q2)

# Q3: Average sensor readings per case
print('\n📌 Q3: Average sensor readings per disease group:')
q3 = pd.read_sql_query(
    "SELECT case_type, sensor_type, ROUND(AVG(value),2) as avg_value, "
    "       ROUND(MIN(value),2) as min_val, ROUND(MAX(value),2) as max_val "
    "FROM readings WHERE is_valid=1 GROUP BY case_type, sensor_type "
    "ORDER BY case_type, sensor_type",
    conn
)
display(q3)

# Q4: Invalid record breakdown
print('\n📌 Q4: Validation failure breakdown:')
q4 = pd.read_sql_query(
    "SELECT SUBSTR(error_reason, 1, 30) as reason, COUNT(*) as count "
    "FROM readings WHERE is_valid=0 GROUP BY reason ORDER BY count DESC",
    conn
)
display(q4)

conn.close()
print('\n✅ All queries executed successfully.')

## Section 14: Summary & Conclusions

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# FINAL SUMMARY
# ─────────────────────────────────────────────────────────────────────────────

conn = sqlite3.connect(StorageSystem.DB_PATH)
total_db = pd.read_sql_query("SELECT COUNT(*) as c FROM readings", conn).iloc[0,0]
total_valid = pd.read_sql_query("SELECT COUNT(*) as c FROM readings WHERE is_valid=1", conn).iloc[0,0]
critical_count = pd.read_sql_query("SELECT COUNT(*) as c FROM patient_risk WHERE risk_level='Critical'", conn).iloc[0,0]
warning_count  = pd.read_sql_query("SELECT COUNT(*) as c FROM patient_risk WHERE risk_level='Warning'", conn).iloc[0,0]
normal_count   = pd.read_sql_query("SELECT COUNT(*) as c FROM patient_risk WHERE risk_level='Normal'", conn).iloc[0,0]
conn.close()

print("""
╔══════════════════════════════════════════════════════════════╗
║        🏥  TELEMEDICINE SYSTEM — FINAL SUMMARY               ║
╠══════════════════════════════════════════════════════════════╣
""")

print(f"  ✅ System Requirements Fulfilled:")
print(f"     A. Multi-Patient Reception : 50 patients monitored simultaneously")
print(f"     B. IoT-Style Data Format   : patient_id, case_type, sensor, timestamp, value")
print(f"     C. Data Storage            : CSV + SQLite DB + JSON log")
print(f"     D. Concurrent Processing   : {SYSTEM.num_consumers} consumer threads + 50 producer threads")
print(f"     E. Real-Time Validation    : Missing / Impossible / Out-of-range / Wrong-param")
print(f"     F. Patient Risk Levels     : Normal / Warning / Critical")
print(f"     G. Telemedicine Output     : Full patient report per patient")
print(f"     H. Dashboard              : See dashboard.py (Streamlit)")
print()
print(f"  📊 Data Statistics:")
print(f"     Total records stored : {total_db:,}")
print(f"     Valid records        : {total_valid:,}  ({100*total_valid/total_db:.1f}%)")
print(f"     Invalid records      : {total_db-total_valid:,}  ({100*(total_db-total_valid)/total_db:.1f}%)")
print()
print(f"  🚨 Patient Risk Summary:")
print(f"     🔴 Critical  : {critical_count} patients")
print(f"     🟡 Warning   : {warning_count} patients")
print(f"     🟢 Normal    : {normal_count} patients")
print()
print(f"  💾 Output Files:")
print(f"     telemedicine_data.csv       — All IoT records")
print(f"     telemedicine.db             — SQLite database")
print(f"     telemedicine_log.json       — JSON event log")
print(f"     telemedicine_dashboard.png  — Analytics chart")
print(f"     sensor_distributions.png    — Sensor histograms")
print(f"     dashboard.py                — Streamlit live dashboard")
print("""
╚══════════════════════════════════════════════════════════════╝
""")